In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

stocks = pd.read_parquet(
    INTERIM_DIR / "combined_nse_ohlcv_raw.parquet"
)

stocks = stocks.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(stocks.shape)
stocks.head()

(121775, 10)


,Date,Adj Close,Close,High,Low,Open,Volume,ticker,company,sector
0,2015-01-01,300.506439,319.549988,322.500000,316.250000,319.000000,1456204,ADANIPORTS.NS,Adani Ports,Industrials
1,2015-01-02,300.318268,319.350006,325.799988,318.049988,319.350006,2894058,ADANIPORTS.NS,Adani Ports,Industrials
2,2015-01-05,304.503113,323.799988,327.500000,319.350006,320.450012,2099786,ADANIPORTS.NS,Adani Ports,Industrials
3,2015-01-06,302.669373,321.850006,331.450012,315.600006,321.649994,3672197,ADANIPORTS.NS,Adani Ports,Industrials
4,2015-01-07,301.964050,321.100006,328.700012,317.399994,321.950012,2981544,ADANIPORTS.NS,Adani Ports,Industrials


In [2]:
PAST_WINDOW = 10
FUTURE_WINDOW = 5

stocks["past_return_10d"] = (
    stocks.groupby("ticker")["Adj Close"]
    .pct_change(PAST_WINDOW)
)

stocks["future_return_5d"] = (
    stocks.groupby("ticker")["Adj Close"]
    .shift(-FUTURE_WINDOW)
    / stocks["Adj Close"]
    - 1
)

stocks[[
    "Date", "ticker", "Adj Close",
    "past_return_10d", "future_return_5d"
]].head(15)

,Date,ticker,Adj Close,past_return_10d,future_return_5d
0,2015-01-01,ADANIPORTS.NS,300.506439,NaN,0.046941
1,2015-01-02,ADANIPORTS.NS,300.318268,NaN,0.036950
2,2015-01-05,ADANIPORTS.NS,304.503113,NaN,0.014361
3,2015-01-06,ADANIPORTS.NS,302.669373,NaN,0.003573
4,2015-01-07,ADANIPORTS.NS,301.964050,NaN,0.031610
5,2015-01-08,ADANIPORTS.NS,314.612488,NaN,-0.011807
6,2015-01-09,ADANIPORTS.NS,311.415070,NaN,-0.000906
7,2015-01-12,ADANIPORTS.NS,308.876038,NaN,-0.000305
8,2015-01-13,ADANIPORTS.NS,303.750793,NaN,0.023220
9,2015-01-14,ADANIPORTS.NS,311.509155,NaN,-0.005736


In [3]:
DECLINE_THRESHOLD = -0.05
REVERSAL_THRESHOLD = 0.03

# A row enters the ML dataset only after a meaningful 10-day decline.
stocks["candidate_decline"] = (
    stocks["past_return_10d"] <= DECLINE_THRESHOLD
)

# Target is defined only for candidate-decline rows.
stocks["reversal"] = np.where(
    stocks["candidate_decline"] & (
        stocks["future_return_5d"] >= REVERSAL_THRESHOLD
    ),
    1,
    0
)

# Keep only rows where both past and future returns are available,
# then keep only candidate-decline events.
labeled_events = stocks.dropna(
    subset=["past_return_10d", "future_return_5d"]
).copy()

labeled_events = labeled_events[
    labeled_events["candidate_decline"]
].copy()

labeled_events["reversal"] = labeled_events["reversal"].astype(int)

print("Candidate decline events:", len(labeled_events))
print("\nClass counts:")
print(labeled_events["reversal"].value_counts())

print("\nClass percentages:")
print(
    labeled_events["reversal"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Candidate decline events: 15074

Class counts:
reversal
0    11179
1     3895
Name: count, dtype: int64

Class percentages:
reversal
0    74.16
1    25.84
Name: proportion, dtype: float64


In [4]:
label_by_stock = (
    labeled_events.groupby("ticker")
    .agg(
        candidate_events=("reversal", "size"),
        reversals=("reversal", "sum"),
        reversal_rate=("reversal", "mean")
    )
)

label_by_stock["reversal_rate"] = (
    label_by_stock["reversal_rate"] * 100
).round(2)

label_by_stock.sort_values(
    "candidate_events",
    ascending=False
).head(15)

,candidate_events,reversals,reversal_rate
ticker,,,
HINDALCO.NS,521,156,29.94
TATASTEEL.NS,488,148,30.33
SBIN.NS,478,109,22.80
ADANIPORTS.NS,470,155,32.98
INDUSINDBK.NS,445,127,28.54
ONGC.NS,442,114,25.79
EICHERMOT.NS,424,148,34.91
BPCL.NS,412,123,29.85
AXISBANK.NS,405,110,27.16


In [5]:
examples = labeled_events[
    [
        "Date", "ticker", "company", "sector",
        "Adj Close", "past_return_10d",
        "future_return_5d", "reversal"
    ]
].copy()

examples["past_return_10d"] = (
    examples["past_return_10d"] * 100
).round(2)

examples["future_return_5d"] = (
    examples["future_return_5d"] * 100
).round(2)

examples.head(20)

,Date,ticker,company,sector,Adj Close,past_return_10d,future_return_5d,reversal
24,2015-02-05,ADANIPORTS.NS,Adani Ports,Industrials,293.594452,-5.21,7.82,1
25,2015-02-06,ADANIPORTS.NS,Adani Ports,Industrials,282.309601,-8.86,11.71,1
26,2015-02-09,ADANIPORTS.NS,Adani Ports,Industrials,282.262573,-13.48,10.36,1
27,2015-02-10,ADANIPORTS.NS,Adani Ports,Industrials,298.907745,-7.91,5.02,1
46,2015-03-11,ADANIPORTS.NS,Adani Ports,Industrials,289.456665,-7.86,2.88,0
47,2015-03-12,ADANIPORTS.NS,Adani Ports,Industrials,291.760681,-5.47,0.85,0
48,2015-03-13,ADANIPORTS.NS,Adani Ports,Industrials,287.999023,-6.43,0.62,0
49,2015-03-16,ADANIPORTS.NS,Adani Ports,Industrials,294.628876,-9.52,-3.02,0
51,2015-03-18,ADANIPORTS.NS,Adani Ports,Industrials,297.779236,-5.91,-0.60,0
52,2015-03-19,ADANIPORTS.NS,Adani Ports,Industrials,294.252747,-5.75,-2.78,0


In [6]:
labeled_events.to_parquet(
    PROCESSED_DIR / "nse_reversal_events_v1.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "nse_reversal_events_v1.parquet")

Saved: /Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/processed/nse_reversal_events_v1.parquet
